# 06.2 — Consolidação da Decisão Operacional

Este notebook formaliza a decisão operacional antes do acesso ao conjunto **out-of-time (OOT)**.

## Objetivo

Registrar, de forma reproduzível, a regra que será congelada antes da avaliação final:

- abordagem operacional baseada em **Top-k** ou em **threshold**;
- capacidade operacional ou ponto de corte escolhido;
- modelo(s) que seguirão para a avaliação final;
- justificativa da decisão;
- data e origem da decisão.

## Regra metodológica

Este notebook **não acessa o OOT** e **não retreina modelos**.

A decisão deve ser estabelecida a partir:

1. dos resultados do período de teste/validação operacional;
2. da capacidade operacional do negócio;
3. da discussão com especialistas;
4. dos objetivos definidos para a classe positiva `Resolvida`.

Somente após a decisão ser registrada e congelada deverá ser executado o notebook de avaliação OOT.


## 1. Bibliotecas e diretórios


In [19]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROJECT_DIR = Path(r"C:\UnB\script_md")
TABLES_DIR = PROJECT_DIR / "outputs" / "tabelas"
CONFIG_DIR = PROJECT_DIR / "outputs" / "config"

CONFIG_DIR.mkdir(parents=True, exist_ok=True)

FILE_TOPK = TABLES_DIR / "operacional_precision_recall_lift_topk.csv"
FILE_THRESHOLD = TABLES_DIR / "operacional_metricas_por_threshold.csv"
FILE_MODELOS_TESTE = TABLES_DIR / "modelagem_comparacao_teste.csv"

for arquivo in [FILE_TOPK, FILE_THRESHOLD, FILE_MODELOS_TESTE]:
    if not arquivo.exists():
        raise FileNotFoundError(
            f"Arquivo não encontrado: {arquivo}. "
            "Execute primeiro os notebooks 06 e 06.1."
        )


## 2. Carregamento das evidências já produzidas


In [20]:
topk = pd.read_csv(
    FILE_TOPK,
    sep=";",
    encoding="utf-8-sig"
)

thresholds = pd.read_csv(
    FILE_THRESHOLD,
    sep=";",
    encoding="utf-8-sig"
)

resultados_teste = pd.read_csv(
    FILE_MODELOS_TESTE,
    sep=";",
    encoding="utf-8-sig"
)

print(f"Resultados Top-k: {len(topk)} linhas")
print(f"Resultados por threshold: {len(thresholds)} linhas")
print(f"Modelos comparados no teste: {len(resultados_teste)}")


Resultados Top-k: 20 linhas
Resultados por threshold: 364 linhas
Modelos comparados no teste: 4


## 3. Resumo dos cenários operacionais

Como referência para a discussão com especialistas, são apresentados os cenários de 10%, 20% e 30% de capacidade operacional.


In [21]:
cenarios_prioritarios = (
    topk[
        topk["Top (%)"].isin([10.0, 20.0, 30.0])
    ]
    .copy()
    .sort_values(
        ["Top (%)", "Precision@k"],
        ascending=[True, False]
    )
)

display(
    cenarios_prioritarios[
        [
            "Modelo",
            "Top (%)",
            "Selecionadas",
            "Precision@k",
            "Recall@k",
            "Lift@k"
        ]
    ].round(4)
)


,Modelo,Top (%),Selecionadas,Precision@k,Recall@k,Lift@k
16,Gradient Boosting,10.0000,1403,0.6379,0.1659,1.6586
11,Random Forest,10.0000,1403,0.6329,0.1646,1.6456
1,Regressão Logística,10.0000,1403,0.6230,0.1620,1.6197
6,KNN,10.0000,1403,0.6058,0.1576,1.5752
12,Random Forest,20.0000,2806,0.5773,0.3003,1.5011
2,Regressão Logística,20.0000,2806,0.5731,0.2981,1.4900
17,Gradient Boosting,20.0000,2806,0.5691,0.2960,1.4798
7,KNN,20.0000,2806,0.5396,0.2806,1.4029
3,Regressão Logística,30.0000,4209,0.5412,0.4222,1.4072
13,Random Forest,30.0000,4209,0.5410,0.4221,1.4066


## 4. Registro da decisão dos especialistas

### Preencha somente após a discussão com o negócio

Escolha **uma** das duas estratégias:

- `"top_k"`: priorização por capacidade operacional;
- `"threshold"`: classificação por ponto de corte probabilístico.

Se ainda não houver decisão, mantenha `None`. O notebook não permitirá congelar a configuração.


In [22]:
# ============================================================
# PREENCHER APÓS A DECISÃO COM OS ESPECIALISTAS
# ============================================================

ESTRATEGIA_OPERACIONAL = "threshold"
# Exemplos:
# ESTRATEGIA_OPERACIONAL = "top_k"
# ESTRATEGIA_OPERACIONAL = "threshold"

TOP_K_PERCENTUAL = None
# Exemplo: 10.0, 20.0 ou 30.0
# Preencher somente se ESTRATEGIA_OPERACIONAL == "top_k"

THRESHOLD_OPERACIONAL = 0.60
# Exemplo: 0.60
# Preencher somente se ESTRATEGIA_OPERACIONAL == "threshold"

MODELOS_PARA_OOT = [
    "Regressão Logística",
    "KNN",
    "Random Forest",
    "Gradient Boosting"
]
# Por padrão, os quatro modelos seguem para a avaliação científica no OOT.
# Alterar somente se houver justificativa metodológica explícita.

JUSTIFICATIVA_NEGOCIO = "Especialistas querem ter mais certeza das que serão resolvidas."
# Exemplo:
# "Especialistas indicaram capacidade operacional aproximada de 20% das reclamações."

DATA_DECISAO = "2026-09-24"
# Exemplo: "2026-09-25"

FONTE_DECISAO = "Consulta com dois especialistas da Ouvidoria"
# Exemplo: "Consulta com dois especialistas da Ouvidoria"


## 5. Validação da decisão

O bloco abaixo verifica se a regra foi preenchida de forma consistente antes de permitir seu congelamento.


In [23]:
def validar_decisao():
    erros = []

    if ESTRATEGIA_OPERACIONAL not in {"top_k", "threshold"}:
        erros.append(
            "ESTRATEGIA_OPERACIONAL deve ser 'top_k' ou 'threshold'."
        )

    if ESTRATEGIA_OPERACIONAL == "top_k":
        if TOP_K_PERCENTUAL is None:
            erros.append(
                "TOP_K_PERCENTUAL deve ser preenchido para estratégia top_k."
            )
        elif not (0 < float(TOP_K_PERCENTUAL) <= 100):
            erros.append(
                "TOP_K_PERCENTUAL deve estar entre 0 e 100."
            )

    if ESTRATEGIA_OPERACIONAL == "threshold":
        if THRESHOLD_OPERACIONAL is None:
            erros.append(
                "THRESHOLD_OPERACIONAL deve ser preenchido para estratégia threshold."
            )
        elif not (0 < float(THRESHOLD_OPERACIONAL) < 1):
            erros.append(
                "THRESHOLD_OPERACIONAL deve estar entre 0 e 1."
            )

    if not MODELOS_PARA_OOT:
        erros.append(
            "MODELOS_PARA_OOT não pode estar vazio."
        )

    if JUSTIFICATIVA_NEGOCIO is None:
        erros.append(
            "JUSTIFICATIVA_NEGOCIO deve ser preenchida."
        )

    if DATA_DECISAO is None:
        erros.append(
            "DATA_DECISAO deve ser preenchida."
        )

    if FONTE_DECISAO is None:
        erros.append(
            "FONTE_DECISAO deve ser preenchida."
        )

    return erros


erros = validar_decisao()

if erros:
    print("Decisão ainda NÃO congelada.")
    print()
    for erro in erros:
        print(f"- {erro}")
else:
    print("Decisão preenchida e pronta para congelamento.")


Decisão preenchida e pronta para congelamento.


## 6. Evidência correspondente à regra escolhida

Após o preenchimento da decisão, este bloco recupera os resultados observados no período de validação operacional para documentar o ponto escolhido.


In [24]:
def obter_evidencia_decisao():
    if ESTRATEGIA_OPERACIONAL == "top_k":
        evidencia = topk[
            topk["Top (%)"] == float(TOP_K_PERCENTUAL)
        ].copy()

        if evidencia.empty:
            raise ValueError(
                f"Não há resultados calculados para Top {TOP_K_PERCENTUAL}%."
            )

        return evidencia[
            [
                "Modelo",
                "Top (%)",
                "Selecionadas",
                "Precision@k",
                "Recall@k",
                "Lift@k"
            ]
        ]

    if ESTRATEGIA_OPERACIONAL == "threshold":
        evidencia = thresholds[
            thresholds["Threshold"].round(4) ==
            round(float(THRESHOLD_OPERACIONAL), 4)
        ].copy()

        if evidencia.empty:
            raise ValueError(
                f"Não há resultados calculados para threshold {THRESHOLD_OPERACIONAL}."
            )

        evidencia["Cobertura (%)"] = evidencia["Cobertura"] * 100

        return evidencia[
            [
                "Modelo",
                "Threshold",
                "Precisão",
                "Revocação",
                "F1-score",
                "Selecionadas",
                "Cobertura (%)"
            ]
        ]

    return None


if not erros:
    evidencia_decisao = obter_evidencia_decisao()
    display(evidencia_decisao.round(4))
else:
    print("Preencha e valide a decisão antes de exibir a evidência final.")


,Modelo,Threshold,Precisão,Revocação,F1-score,Selecionadas,Cobertura (%)
55,Regressão Logística,0.6000,0.5830,0.2532,0.3531,2343,16.7035
146,KNN,0.6000,0.5596,0.2221,0.3179,2141,15.2634
237,Random Forest,0.6000,0.6328,0.1687,0.2664,1438,10.2517
328,Gradient Boosting,0.6000,0.6212,0.1952,0.2970,1695,12.0838


## 7. Congelamento da configuração

A configuração somente será salva quando todos os campos obrigatórios estiverem preenchidos.

O arquivo gerado será utilizado pelo notebook `07_avaliacao_OOT.ipynb`.


In [25]:
if erros:
    print(
        "Configuração NÃO salva. "
        "Preencha os campos da Seção 4 e execute novamente as Seções 5 a 7."
    )
else:
    configuracao = {
        "estrategia_operacional": ESTRATEGIA_OPERACIONAL,
        "top_k_percentual": TOP_K_PERCENTUAL,
        "threshold_operacional": THRESHOLD_OPERACIONAL,
        "modelos_para_oot": MODELOS_PARA_OOT,
        "justificativa_negocio": JUSTIFICATIVA_NEGOCIO,
        "data_decisao": DATA_DECISAO,
        "fonte_decisao": FONTE_DECISAO,
        "regra_congelada": True,
        "oot_acessado_nesta_etapa": False
    }

    CONFIG_FILE = CONFIG_DIR / "configuracao_operacional_congelada.json"

    with CONFIG_FILE.open(
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            configuracao,
            f,
            ensure_ascii=False,
            indent=2
        )

    print(f"Configuração congelada e salva em:")
    print(CONFIG_FILE)


Configuração congelada e salva em:
C:\UnB\script_md\outputs\config\configuracao_operacional_congelada.json


## 8. Registro para o artigo

Após a definição da regra operacional, a metodologia poderá registrar que:

> Após a comparação inicial dos classificadores, o período intermediário foi utilizado para examinar o comportamento operacional das probabilidades estimadas. Foram analisados diferentes thresholds e níveis de capacidade de priorização por meio de Precision@k, Recall@k e Lift@k. O ponto operacional foi definido considerando conjuntamente o desempenho empírico e a capacidade de atuação informada pelos especialistas, sendo posteriormente congelado antes do acesso ao conjunto out-of-time.

A redação definitiva deverá incorporar a regra efetivamente escolhida.


## 9. Próxima etapa

Somente após a criação de:

`outputs/config/configuracao_operacional_congelada.json`

deverá ser executado:

`07_avaliacao_OOT.ipynb`

No OOT, nenhuma decisão de desenvolvimento poderá ser alterada com base nos resultados observados.
